# Detección de cumplimiento de EPP en obras de construcción con YOLOv8

Cuaderno completo del proyecto: detecta a los trabajadores y su Equipo de Protección Personal
(casco y chaleco) y verifica el cumplimiento por persona. Todo el código está incluido en el
propio cuaderno; al ejecutarlo en orden se reproduce el proyecto completo: descarga del dataset,
entrenamiento, evaluación, verificador de cumplimiento y análisis de robustez.

Antes de empezar:
1. Activar la GPU en `Entorno de ejecución → Cambiar tipo de entorno → GPU`.
2. Tener una clave de API de Roboflow (cuenta gratuita) para descargar el dataset.


## 1. Entorno y dependencias

In [ ]:
!nvidia-smi

In [ ]:
!pip install -q ultralytics roboflow python-dotenv pyyaml
import torch; print("Torch", torch.__version__, "| CUDA:", torch.cuda.is_available())
import os
os.makedirs("src", exist_ok=True)


## 2. Clave de API de Roboflow

In [ ]:
ROBOFLOW_API_KEY = ""  #@param {type:"string"}
open(".env","w").write(f"ROBOFLOW_API_KEY={ROBOFLOW_API_KEY}\n")
assert ROBOFLOW_API_KEY, "Pega tu Private API Key de Roboflow y vuelve a ejecutar."


## 3. Código fuente

### 3.0 Módulo del verificador (`compliance_checker.py`)

In [ ]:
%%writefile src/compliance_checker.py
"""
Verificador de cumplimiento de EPP basado en reglas (APORTE PROPIO del proyecto).

YOLO detecta objetos sueltos (Person, Hardhat, Safety Vest, ...). Este módulo aplica una
lógica geométrica para decidir, POR CADA PERSONA, si cumple con el EPP, y calcula la tasa
de cumplimiento de cada fotograma.

Diseño:
  - Es independiente del modelo: opera sobre detecciones genéricas (clase, confianza, caja).
    Así se puede testear sin YOLO (ver self-test en __main__) y reutilizar en imágenes/video.

Regla de cumplimiento:
  Una persona CUMPLE si tiene casco (Hardhat) en su región de cabeza Y chaleco (Safety Vest)
  en su torso, y no presenta violaciones explícitas (NO-Hardhat / NO-Safety Vest).
  Criterio conservador (pro-seguridad): si falta evidencia de un EPP, se marca NO conforme.

Limitaciones (para el informe): la asociación es geométrica y depende de la calidad del
detector y de los umbrales de región; no maneja perspectiva 3D ni oclusión total.
"""

from dataclasses import dataclass, field

# Nombres de clase relevantes del dataset Construction Site Safety
CLS_PERSON = "Person"
CLS_HARDHAT = "Hardhat"
CLS_NO_HARDHAT = "NO-Hardhat"
CLS_VEST = "Safety Vest"
CLS_NO_VEST = "NO-Safety Vest"


@dataclass
class Detection:
    """Una detección genérica de YOLO (en píxeles, formato xyxy)."""
    cls_name: str
    conf: float
    xyxy: tuple  # (x1, y1, x2, y2)


@dataclass
class PersonStatus:
    """Resultado del análisis de cumplimiento de una persona."""
    box: tuple
    has_hardhat: bool = False
    has_vest: bool = False
    helmet_violation: bool = False   # se detectó NO-Hardhat
    vest_violation: bool = False     # se detectó NO-Safety Vest
    missing: list = field(default_factory=list)  # qué EPP falta

    @property
    def compliant(self) -> bool:
        return (
            self.has_hardhat
            and self.has_vest
            and not self.helmet_violation
            and not self.vest_violation
        )


def _center(box: tuple) -> tuple:
    x1, y1, x2, y2 = box
    return ((x1 + x2) / 2, (y1 + y2) / 2)


def _area(box: tuple) -> float:
    x1, y1, x2, y2 = box
    return max(0.0, x2 - x1) * max(0.0, y2 - y1)


def _point_in_box(point: tuple, box: tuple) -> bool:
    px, py = point
    x1, y1, x2, y2 = box
    return x1 <= px <= x2 and y1 <= py <= y2


class PPEComplianceChecker:
    """Aplica la regla de cumplimiento sobre las detecciones de un fotograma."""

    def __init__(
        self,
        min_conf: float = 0.25,
        head_frac: float = 0.40,   # región de cabeza: 40% superior de la persona
        torso_top: float = 0.20,   # torso: desde 20%...
        torso_bot: float = 0.75,   # ...hasta 75% de la altura de la persona
    ):
        self.min_conf = min_conf
        self.head_frac = head_frac
        self.torso_top = torso_top
        self.torso_bot = torso_bot

    def _assign_person(self, item_center: tuple, persons: list) -> int:
        """Devuelve el índice de la persona cuya caja contiene el centro del EPP.
        Si varias lo contienen, elige la de menor área (la más específica/cercana)."""
        best_idx, best_area = -1, float("inf")
        for i, p in enumerate(persons):
            if _point_in_box(item_center, p.xyxy):
                a = _area(p.xyxy)
                if a < best_area:
                    best_idx, best_area = i, a
        return best_idx

    def _in_head_region(self, center: tuple, person_box: tuple) -> bool:
        x1, y1, x2, y2 = person_box
        h = y2 - y1
        return y1 <= center[1] <= y1 + self.head_frac * h

    def _in_torso_region(self, center: tuple, person_box: tuple) -> bool:
        x1, y1, x2, y2 = person_box
        h = y2 - y1
        return y1 + self.torso_top * h <= center[1] <= y1 + self.torso_bot * h

    def check_frame(self, detections: list):
        """Analiza un fotograma.
        Args:
            detections: lista de Detection.
        Returns:
            (statuses, compliance_rate) — lista de PersonStatus y la tasa [0..1] o None
            si no hay personas.
        """
        dets = [d for d in detections if d.conf >= self.min_conf]
        persons = [d for d in dets if d.cls_name == CLS_PERSON]
        statuses = [PersonStatus(box=p.xyxy) for p in persons]

        for d in dets:
            if d.cls_name == CLS_PERSON:
                continue
            c = _center(d.xyxy)
            idx = self._assign_person(c, persons)
            if idx < 0:
                continue  # EPP no asociado a ninguna persona
            st = statuses[idx]
            pbox = persons[idx].xyxy
            if d.cls_name == CLS_HARDHAT and self._in_head_region(c, pbox):
                st.has_hardhat = True
            elif d.cls_name == CLS_NO_HARDHAT and self._in_head_region(c, pbox):
                st.helmet_violation = True
            elif d.cls_name == CLS_VEST and self._in_torso_region(c, pbox):
                st.has_vest = True
            elif d.cls_name == CLS_NO_VEST and self._in_torso_region(c, pbox):
                st.vest_violation = True

        # Anotar qué falta (para reportes)
        for st in statuses:
            if not st.has_hardhat:
                st.missing.append("casco")
            if not st.has_vest:
                st.missing.append("chaleco")

        if not statuses:
            return statuses, None
        n_ok = sum(1 for st in statuses if st.compliant)
        return statuses, n_ok / len(statuses)


def annotate_frame(image, statuses, compliance_rate):
    """Dibuja sobre la imagen: cajas verdes (cumple) / rojas (no cumple) + banner con la tasa.

    Requiere OpenCV. `image` es un array BGR (como lo entrega cv2.imread / frame de video).
    Devuelve la imagen anotada.
    """
    import cv2

    GREEN = (0, 180, 0)
    RED = (0, 0, 255)

    for st in statuses:
        x1, y1, x2, y2 = map(int, st.box)
        color = GREEN if st.compliant else RED
        cv2.rectangle(image, (x1, y1), (x2, y2), color, 2)
        if st.compliant:
            label = "OK"
        else:
            label = "NO: falta " + ",".join(st.missing) if st.missing else "NO conforme"
        cv2.putText(image, label, (x1, max(y1 - 6, 14)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.55, color, 2, cv2.LINE_AA)

    # Banner superior con la tasa de cumplimiento del frame
    n_total = len(statuses)
    n_ok = sum(1 for st in statuses if st.compliant)
    if compliance_rate is None:
        text = "Sin personas detectadas"
    else:
        text = f"Cumplimiento: {compliance_rate:.0%}  ({n_ok}/{n_total})"
    cv2.rectangle(image, (0, 0), (image.shape[1], 32), (0, 0, 0), -1)
    cv2.putText(image, text, (10, 22), cv2.FONT_HERSHEY_SIMPLEX, 0.7,
                (255, 255, 255), 2, cv2.LINE_AA)
    return image


# =====================================================================================
# Self-test: valida la lógica con cajas SINTÉTICAS (no necesita el modelo entrenado)
# =====================================================================================
def _self_test() -> None:
    checker = PPEComplianceChecker(min_conf=0.25)

    # Escenario: 2 personas.
    #  Persona A (x 0-100): tiene casco (arriba) y chaleco (torso) -> CUMPLE
    #  Persona B (x 200-300): tiene NO-Hardhat (arriba), sin chaleco -> NO CUMPLE
    dets = [
        Detection(CLS_PERSON, 0.9, (0, 0, 100, 400)),
        Detection(CLS_HARDHAT, 0.8, (30, 10, 70, 60)),       # cabeza de A
        Detection(CLS_VEST, 0.8, (20, 150, 80, 280)),        # torso de A
        Detection(CLS_PERSON, 0.9, (200, 0, 300, 400)),
        Detection(CLS_NO_HARDHAT, 0.8, (230, 10, 270, 60)),  # cabeza de B (violación)
    ]
    statuses, rate = checker.check_frame(dets)

    assert len(statuses) == 2, "Debe haber 2 personas"
    assert statuses[0].compliant is True, "Persona A debería CUMPLIR"
    assert statuses[1].compliant is False, "Persona B NO debería cumplir"
    assert statuses[1].helmet_violation is True, "Persona B tiene NO-Hardhat"
    assert abs(rate - 0.5) < 1e-9, f"Tasa esperada 0.5, obtenida {rate}"

    print("Self-test OK ✅")
    print(f"  Persona A -> cumple={statuses[0].compliant} (casco={statuses[0].has_hardhat}, chaleco={statuses[0].has_vest})")
    print(f"  Persona B -> cumple={statuses[1].compliant} (violación casco={statuses[1].helmet_violation}, falta={statuses[1].missing})")
    print(f"  Tasa de cumplimiento del frame: {rate:.0%}")


if __name__ == "__main__":
    _self_test()


### 3.1 Descarga del dataset

In [ ]:
%%writefile src/01_download_dataset.py
"""
Descarga el dataset Construction Site Safety desde Roboflow Universe.

La clave de API se lee desde el archivo .env (variable ROBOFLOW_API_KEY) y el dataset se
guarda en datasets/.

Dataset: roboflow-universe-projects/construction-site-safety (v27, formato YOLOv8).
Clases (10): Hardhat, Mask, NO-Hardhat, NO-Mask, NO-Safety Vest, Person, Safety Cone,
             Safety Vest, machinery, vehicle.
"""

import os
import sys
from pathlib import Path

from dotenv import load_dotenv

# --- Configuración del dataset (pública, de Roboflow Universe) ---
WORKSPACE = "roboflow-universe-projects"
PROJECT = "construction-site-safety"
VERSION = 27
FORMAT = "yolov8"

# Carpeta raíz del proyecto (un nivel arriba de src/)
PROJECT_ROOT = Path(__file__).resolve().parent.parent
DOWNLOAD_DIR = PROJECT_ROOT / "datasets"


def main() -> None:
    # Cargar variables del archivo .env
    load_dotenv(PROJECT_ROOT / ".env")
    api_key = os.getenv("ROBOFLOW_API_KEY")

    if not api_key or api_key == "tu_api_key_aqui":
        sys.exit(
            "ERROR: falta ROBOFLOW_API_KEY. Define la variable en un archivo .env "
            "(ver .env.example) con tu clave de API de Roboflow."
        )

    # Importar aquí para que el mensaje de error de arriba salga aunque falte la lib
    from roboflow import Roboflow

    DOWNLOAD_DIR.mkdir(exist_ok=True)

    print(f"Descargando {WORKSPACE}/{PROJECT} v{VERSION} (formato {FORMAT})...")
    rf = Roboflow(api_key=api_key)
    project = rf.workspace(WORKSPACE).project(PROJECT)
    version = project.version(VERSION)
    dataset = version.download(FORMAT, location=str(DOWNLOAD_DIR / PROJECT))

    print("\nListo. Dataset descargado en:")
    print(f"  {dataset.location}")
    print("\nRevisa el archivo data.yaml dentro de esa carpeta.")


if __name__ == "__main__":
    main()


### 3.2 Análisis del dataset (desbalance de clases)

In [ ]:
%%writefile src/02_analyze_dataset.py
"""
Cuantifica el desbalance de clases del dataset: cuenta las instancias (objetos) de cada
clase en cada partición (train/valid/test).

Genera:
    - Tabla por clase impresa en consola, con el ratio de desbalance.
    - outputs/metrics/class_distribution.csv
    - outputs/figures/class_distribution.png
"""

from collections import Counter
from pathlib import Path

import matplotlib

matplotlib.use("Agg")  # backend sin ventana (guardamos a archivo, no mostramos)
import matplotlib.pyplot as plt
import yaml

PROJECT_ROOT = Path(__file__).resolve().parent.parent
DATASET_DIR = PROJECT_ROOT / "datasets" / "construction-site-safety"
DATA_YAML = DATASET_DIR / "data.yaml"
SPLITS = ["train", "valid", "test"]


def count_instances_per_class(labels_dir: Path, num_classes: int) -> Counter:
    """Cuenta las instancias de cada clase en una carpeta de labels YOLO."""
    counts: Counter = Counter()
    if not labels_dir.is_dir():
        return counts
    for txt in labels_dir.glob("*.txt"):
        for line in txt.read_text().splitlines():
            line = line.strip()
            if not line:
                continue
            class_id = int(line.split()[0])  # primer número = id de clase
            counts[class_id] += 1
    return counts


def main() -> None:
    names = yaml.safe_load(DATA_YAML.read_text())["names"]
    num_classes = len(names)

    # Conteo por split
    per_split = {}
    for split in SPLITS:
        labels_dir = DATASET_DIR / split / "labels"
        per_split[split] = count_instances_per_class(labels_dir, num_classes)

    # --- Tabla en consola ---
    header = f"{'id':>2}  {'clase':<16}{'train':>8}{'valid':>8}{'test':>8}{'TOTAL':>9}"
    print(header)
    print("-" * len(header))
    totals = Counter()
    rows = []
    for cid, name in enumerate(names):
        tr = per_split["train"][cid]
        va = per_split["valid"][cid]
        te = per_split["test"][cid]
        tot = tr + va + te
        totals[cid] = tot
        rows.append((cid, name, tr, va, te, tot))
        print(f"{cid:>2}  {name:<16}{tr:>8}{va:>8}{te:>8}{tot:>9}")

    grand_total = sum(totals.values())
    print("-" * len(header))
    print(f"{'':>2}  {'TOTAL':<16}{'':>8}{'':>8}{'':>8}{grand_total:>9}")

    # Métrica de desbalance: ratio entre la clase más y menos frecuente
    sorted_tot = sorted(totals.values(), reverse=True)
    if sorted_tot[-1] > 0:
        ratio = sorted_tot[0] / sorted_tot[-1]
        print(f"\nDesbalance (clase mayoritaria / minoritaria): {ratio:.1f}x")

    # --- Guardar CSV ---
    out_metrics = PROJECT_ROOT / "outputs" / "metrics"
    out_metrics.mkdir(parents=True, exist_ok=True)
    csv_path = out_metrics / "class_distribution.csv"
    with csv_path.open("w") as f:
        f.write("class_id,class_name,train,valid,test,total\n")
        for cid, name, tr, va, te, tot in rows:
            f.write(f"{cid},{name},{tr},{va},{te},{tot}\n")
    print(f"\nCSV guardado en: {csv_path}")

    # --- Guardar gráfico de barras (total por clase) ---
    out_figures = PROJECT_ROOT / "outputs" / "figures"
    out_figures.mkdir(parents=True, exist_ok=True)
    fig, ax = plt.subplots(figsize=(10, 5))
    labels = [r[1] for r in rows]
    values = [r[5] for r in rows]
    ax.bar(labels, values, color="steelblue")
    ax.set_ylabel("Número de instancias (total)")
    ax.set_title("Distribución de clases — Construction Site Safety")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    fig_path = out_figures / "class_distribution.png"
    fig.savefig(fig_path, dpi=120)
    print(f"Gráfico guardado en: {fig_path}")


if __name__ == "__main__":
    main()


### 3.3 Verificación de anotaciones

In [ ]:
%%writefile src/03_visualize_samples.py
"""
Verificación visual de las anotaciones: dibuja algunas imágenes de entrenamiento con sus
cajas (ground truth) para confirmar que las etiquetas están bien alineadas con las imágenes.

Genera: outputs/figures/sample_annotations.png (mosaico de imágenes anotadas).
"""

import random
from pathlib import Path

import cv2
import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import yaml

PROJECT_ROOT = Path(__file__).resolve().parent.parent
DATASET_DIR = PROJECT_ROOT / "datasets" / "construction-site-safety"
DATA_YAML = DATASET_DIR / "data.yaml"
N_SAMPLES = 6
SEED = 42  # fijo para reproducibilidad

# Un color por clase (BGR), generado de forma determinista
def color_for(class_id: int) -> tuple:
    rng = random.Random(class_id * 7 + 13)
    return (rng.randint(50, 255), rng.randint(50, 255), rng.randint(50, 255))


def draw_boxes(img, label_path: Path, names: list):
    """Dibuja las cajas YOLO (normalizadas) sobre la imagen."""
    h, w = img.shape[:2]
    if not label_path.exists():
        return img
    for line in label_path.read_text().splitlines():
        parts = line.split()
        if len(parts) < 5:
            continue
        cid = int(parts[0])
        xc, yc, bw, bh = map(float, parts[1:5])
        x1 = int((xc - bw / 2) * w)
        y1 = int((yc - bh / 2) * h)
        x2 = int((xc + bw / 2) * w)
        y2 = int((yc + bh / 2) * h)
        color = color_for(cid)
        cv2.rectangle(img, (x1, y1), (x2, y2), color, 2)
        cv2.putText(img, names[cid], (x1, max(y1 - 5, 12)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 1, cv2.LINE_AA)
    return img


def main() -> None:
    names = yaml.safe_load(DATA_YAML.read_text())["names"]
    images_dir = DATASET_DIR / "train" / "images"
    labels_dir = DATASET_DIR / "train" / "labels"

    all_images = sorted(images_dir.glob("*.jpg"))
    random.Random(SEED).shuffle(all_images)
    samples = all_images[:N_SAMPLES]

    cols = 3
    rows = (N_SAMPLES + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(15, 5 * rows))
    axes = axes.flatten()

    for ax, img_path in zip(axes, samples):
        img = cv2.imread(str(img_path))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        label_path = labels_dir / (img_path.stem + ".txt")
        img = draw_boxes(img, label_path, names)
        ax.imshow(img)
        ax.set_title(img_path.name[:30], fontsize=8)
        ax.axis("off")

    for ax in axes[len(samples):]:
        ax.axis("off")

    plt.tight_layout()
    out = PROJECT_ROOT / "outputs" / "figures" / "sample_annotations.png"
    out.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(out, dpi=110)
    print(f"Mosaico guardado en: {out}")


if __name__ == "__main__":
    main()


### 3.4 Entrenamiento (YOLOv8s)

In [ ]:
%%writefile src/04_train.py
"""
Entrena un modelo YOLOv8 para la detección de EPP en obras de construcción.

Las decisiones de entrenamiento se definen y documentan en el bloque CONFIG de abajo. Los
resultados se guardan en outputs/training/<name>/: pesos (best.pt / last.pt), curvas de
pérdida, matriz de confusión y curvas PR.
"""

from pathlib import Path

from ultralytics import YOLO

PROJECT_ROOT = Path(__file__).resolve().parent.parent
DATA_YAML = PROJECT_ROOT / "datasets" / "construction-site-safety" / "data.yaml"
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "training"

# =====================================================================================
# CONFIG — decisiones de entrenamiento documentadas
# =====================================================================================
CONFIG = {
    # --- Modelo ---
    "model": "yolov8s.pt",   # YOLOv8 "small", preentrenado en COCO (transfer learning).
                             #   YOLOv8 es ANCHOR-FREE: no se configuran anclas (anchors).

    # --- Datos ---
    "data": str(DATA_YAML),

    # --- Hiperparámetros de entrenamiento ---
    "epochs": 60,            # pasadas completas por el dataset
    "imgsz": 640,            # resolución de entrada (estándar YOLO)
    "batch": 8,              # tamaño de lote — cabe en 8 GB de VRAM (RTX 4070 Laptop)
    "device": 0,             # GPU 0  (usar 'cpu' si no hubiera GPU)
    "patience": 20,          # early stopping: para si no mejora en 20 épocas
    "seed": 0,               # reproducibilidad

    # --- Data augmentation (entregable: documentar mosaic on/off) ---
    "close_mosaic": 10,      # apaga el augmentation "mosaic" en las últimas 10 épocas
                             #   (mejora la precisión fina al final del entrenamiento)

    # --- Dónde guardar ---
    "project": str(OUTPUT_DIR),
    "name": "yolov8s_baseline",
    "exist_ok": True,        # sobrescribe si ya existe la carpeta
}
# =====================================================================================


def main() -> None:
    print(f"Dataset: {DATA_YAML}")
    print(f"Modelo : {CONFIG['model']}  |  epochs={CONFIG['epochs']}  "
          f"batch={CONFIG['batch']}  imgsz={CONFIG['imgsz']}")

    model = YOLO(CONFIG["model"])
    results = model.train(**CONFIG)

    print("\nEntrenamiento terminado.")
    print(f"Resultados en: {OUTPUT_DIR / CONFIG['name']}")
    print("  - weights/best.pt  -> el mejor modelo (usar este para validar/predecir)")
    print("  - results.png, confusion_matrix.png, *_curve.png -> figuras para el informe")


if __name__ == "__main__":
    main()


### 3.5 Evaluación en test (mAP por clase)

In [ ]:
%%writefile src/05_validate.py
"""
Evalúa el modelo entrenado sobre el conjunto de prueba (test): mAP@0.5 y mAP@0.5:0.95
globales y por clase, más la matriz de confusión.

Genera:
    - Tabla por clase impresa en consola.
    - outputs/metrics/test_metrics_per_class.csv
    - outputs/validation/test/ (matriz de confusión, curvas PR, etc.)
"""

from pathlib import Path

from ultralytics import YOLO

PROJECT_ROOT = Path(__file__).resolve().parent.parent
DATA_YAML = PROJECT_ROOT / "datasets" / "construction-site-safety" / "data.yaml"
WEIGHTS = PROJECT_ROOT / "outputs" / "training" / "yolov8s_baseline" / "weights" / "best.pt"
VAL_DIR = PROJECT_ROOT / "outputs" / "validation"
SPLIT = "test"  # evaluar sobre el conjunto de prueba (no visto en entrenamiento)


def main() -> None:
    if not WEIGHTS.exists():
        raise SystemExit(f"No existe el modelo: {WEIGHTS}\nEntrena primero con: python src/04_train.py")

    print(f"Evaluando {WEIGHTS.name} sobre el split '{SPLIT}'...\n")
    model = YOLO(str(WEIGHTS))
    metrics = model.val(
        data=str(DATA_YAML),
        split=SPLIT,
        project=str(VAL_DIR),
        name=SPLIT,
        exist_ok=True,
        plots=True,   # genera matriz de confusión y curvas PR
    )

    # --- Métricas globales ---
    print("\n=== Métricas globales (test) ===")
    print(f"  mAP@0.5     : {metrics.box.map50:.4f}")
    print(f"  mAP@0.5:0.95: {metrics.box.map:.4f}")
    print(f"  Precision   : {metrics.box.mp:.4f}")
    print(f"  Recall      : {metrics.box.mr:.4f}")

    # --- Desglose por clase ---
    names = metrics.names
    print("\n=== Desglose por clase ===")
    header = f"{'clase':<16}{'P':>8}{'R':>8}{'mAP50':>9}{'mAP50-95':>10}"
    print(header)
    print("-" * len(header))

    rows = []
    for i, c in enumerate(metrics.box.ap_class_index):
        name = names[c]
        p = metrics.box.p[i]
        r = metrics.box.r[i]
        ap50 = metrics.box.ap50[i]
        ap = metrics.box.maps[c]  # mAP50-95 de esta clase
        rows.append((name, p, r, ap50, ap))
        print(f"{name:<16}{p:>8.3f}{r:>8.3f}{ap50:>9.3f}{ap:>10.3f}")

    # --- Guardar CSV ---
    out_metrics = PROJECT_ROOT / "outputs" / "metrics"
    out_metrics.mkdir(parents=True, exist_ok=True)
    csv_path = out_metrics / "test_metrics_per_class.csv"
    with csv_path.open("w") as f:
        f.write("class_name,precision,recall,mAP50,mAP50-95\n")
        for name, p, r, ap50, ap in rows:
            f.write(f"{name},{p:.4f},{r:.4f},{ap50:.4f},{ap:.4f}\n")
    print(f"\nCSV por clase: {csv_path}")
    print(f"Matriz de confusión y curvas en: {VAL_DIR / SPLIT}")


if __name__ == "__main__":
    main()


### 3.6 Verificador de cumplimiento sobre imágenes

In [ ]:
%%writefile src/06_predict_images.py
"""
Demostración del verificador de cumplimiento sobre imágenes del conjunto de prueba.
Conecta el modelo entrenado (best.pt) con el módulo compliance_checker.

Flujo: imagen -> YOLO detecta objetos -> compliance_checker decide el cumplimiento por
persona -> se dibujan cajas (verde = cumple, rojo = no) y un banner con la tasa del fotograma.

Genera:
    - outputs/compliance/images/ (imágenes anotadas)
    - outputs/metrics/compliance_per_image.csv
    - report/figures/demo_cumplimiento.png (mosaico para el informe)
"""

from pathlib import Path

import cv2
import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
from ultralytics import YOLO

from compliance_checker import PPEComplianceChecker, Detection, annotate_frame

PROJECT_ROOT = Path(__file__).resolve().parent.parent
WEIGHTS = PROJECT_ROOT / "outputs" / "training" / "yolov8s_baseline" / "weights" / "best.pt"
TEST_IMAGES = PROJECT_ROOT / "datasets" / "construction-site-safety" / "test" / "images"
OUT_DIR = PROJECT_ROOT / "outputs" / "compliance" / "images"
CSV_PATH = PROJECT_ROOT / "outputs" / "metrics" / "compliance_per_image.csv"
MONTAGE_PATH = PROJECT_ROOT / "report" / "figures" / "demo_cumplimiento.png"

CONF = 0.25          # umbral de confianza para las detecciones
MAX_ANNOTATED = 20   # cuántas imágenes (con personas) guardar anotadas
N_MONTAGE = 6        # cuántas poner en el mosaico del informe


def detections_from_result(result, names) -> list:
    """Convierte el resultado de YOLO en una lista de Detection del compliance_checker."""
    dets = []
    for box in result.boxes:
        cls_id = int(box.cls)
        dets.append(Detection(
            cls_name=names[cls_id],
            conf=float(box.conf),
            xyxy=tuple(float(v) for v in box.xyxy[0].tolist()),
        ))
    return dets


def main() -> None:
    if not WEIGHTS.exists():
        raise SystemExit(f"No existe el modelo: {WEIGHTS}\nEntrena primero: python src/04_train.py")

    OUT_DIR.mkdir(parents=True, exist_ok=True)
    CSV_PATH.parent.mkdir(parents=True, exist_ok=True)
    MONTAGE_PATH.parent.mkdir(parents=True, exist_ok=True)

    model = YOLO(str(WEIGHTS))
    checker = PPEComplianceChecker(min_conf=CONF)

    images = sorted(TEST_IMAGES.glob("*.jpg"))
    print(f"Procesando {len(images)} imágenes de test...\n")

    rows = []           # (nombre, n_personas, n_cumplen, tasa)
    annotated_paths = []
    for img_path in images:
        result = model(str(img_path), conf=CONF, verbose=False)[0]
        dets = detections_from_result(result, model.names)
        statuses, rate = checker.check_frame(dets)

        n_persons = len(statuses)
        if n_persons == 0:
            continue  # solo nos interesan frames con personas para la demo de cumplimiento

        n_ok = sum(1 for st in statuses if st.compliant)
        rows.append((img_path.name, n_persons, n_ok, rate))

        if len(annotated_paths) < MAX_ANNOTATED:
            img = cv2.imread(str(img_path))
            img = annotate_frame(img, statuses, rate)
            out = OUT_DIR / img_path.name
            cv2.imwrite(str(out), img)
            annotated_paths.append(out)

    # --- CSV resumen ---
    with CSV_PATH.open("w") as f:
        f.write("imagen,personas,cumplen,tasa_cumplimiento\n")
        for name, n, ok, rate in rows:
            f.write(f"{name},{n},{ok},{rate:.3f}\n")

    # --- Estadísticas globales ---
    total_persons = sum(r[1] for r in rows)
    total_ok = sum(r[2] for r in rows)
    print(f"Imágenes con personas : {len(rows)}")
    print(f"Personas detectadas   : {total_persons}")
    print(f"Personas que cumplen  : {total_ok}")
    if total_persons:
        print(f"Tasa global de cumplimiento: {total_ok / total_persons:.1%}")
    print(f"\nImágenes anotadas en : {OUT_DIR}")
    print(f"CSV por imagen       : {CSV_PATH}")

    # --- Mosaico para el informe ---
    sample = annotated_paths[:N_MONTAGE]
    if sample:
        cols = 3
        rows_n = (len(sample) + cols - 1) // cols
        fig, axes = plt.subplots(rows_n, cols, figsize=(15, 5 * rows_n))
        axes = axes.flatten() if hasattr(axes, "flatten") else [axes]
        for ax, p in zip(axes, sample):
            im = cv2.cvtColor(cv2.imread(str(p)), cv2.COLOR_BGR2RGB)
            ax.imshow(im)
            ax.axis("off")
        for ax in axes[len(sample):]:
            ax.axis("off")
        plt.tight_layout()
        fig.savefig(MONTAGE_PATH, dpi=110)
        print(f"Mosaico informe      : {MONTAGE_PATH}")


if __name__ == "__main__":
    main()


### 3.7 Robustez ante oclusión

In [ ]:
%%writefile src/07_occlusion_analysis.py
"""
Análisis de robustez ante la oclusión, medido por persona.

Un primer intento que agrupaba imágenes por solapamiento de cajas resultó confundido: las
imágenes con más solapamiento eran grupos de personas grandes y claras (más fáciles de
detectar). Para aislar la oclusión, se mide por objeto y solo en la clase Person:

  - Oclusión de una persona = máximo IoU de su caja (ground truth) con cualquier otra caja
    de la imagen (mayor IoU = más tapada por otro objeto).
  - Las personas se agrupan en bajo/medio/alto (terciles de oclusión).
  - Se mide el recall en cada grupo: qué fracción de personas detectó el modelo (una persona
    se considera detectada si existe una predicción Person con IoU >= 0.5).

Responde la pregunta de seguridad: ¿se escapan los trabajadores ocluidos?

Genera:
    - outputs/metrics/occlusion_person_recall.csv
    - report/figures/occlusion_recall.png
"""

from pathlib import Path

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import yaml
from ultralytics import YOLO

PROJECT_ROOT = Path(__file__).resolve().parent.parent
DATASET_DIR = PROJECT_ROOT / "datasets" / "construction-site-safety"
DATA_YAML = DATASET_DIR / "data.yaml"
WEIGHTS = PROJECT_ROOT / "outputs" / "training" / "yolov8s_baseline" / "weights" / "best.pt"
TEST_IMAGES = DATASET_DIR / "test" / "images"
TEST_LABELS = DATASET_DIR / "test" / "labels"

CONF = 0.25
IOU_MATCH = 0.5  # IoU mínimo para considerar una persona "detectada"
GROUPS = ["bajo", "medio", "alto"]


def iou(a, b):
    ax1, ay1, ax2, ay2 = a
    bx1, by1, bx2, by2 = b
    ix1, iy1 = max(ax1, bx1), max(ay1, by1)
    ix2, iy2 = min(ax2, bx2), min(ay2, by2)
    iw, ih = max(0.0, ix2 - ix1), max(0.0, iy2 - iy1)
    inter = iw * ih
    ua = (ax2 - ax1) * (ay2 - ay1) + (bx2 - bx1) * (by2 - by1) - inter
    return inter / ua if ua > 0 else 0.0


def main() -> None:
    if not WEIGHTS.exists():
        raise SystemExit(f"No existe el modelo: {WEIGHTS}")

    names = yaml.safe_load(DATA_YAML.read_text())["names"]
    person_id = names.index("Person")

    model = YOLO(str(WEIGHTS))
    persons = []  # (occlusion, detected)

    for img in sorted(TEST_IMAGES.glob("*.jpg")):
        label = TEST_LABELS / (img.stem + ".txt")
        if not label.exists():
            continue

        result = model(str(img), conf=CONF, verbose=False)[0]
        H, W = result.orig_shape  # alto, ancho en píxeles

        # Cajas GT (todas) en píxeles + cuáles son Person
        gt_boxes, gt_person_idx = [], []
        for line in label.read_text().splitlines():
            p = line.split()
            if len(p) < 5:
                continue
            cid = int(p[0])
            xc, yc, w, h = map(float, p[1:5])
            box = ((xc - w / 2) * W, (yc - h / 2) * H, (xc + w / 2) * W, (yc + h / 2) * H)
            if cid == person_id:
                gt_person_idx.append(len(gt_boxes))
            gt_boxes.append(box)

        # Predicciones Person (píxeles)
        pred_persons = [tuple(b.xyxy[0].tolist()) for b in result.boxes
                        if int(b.cls) == person_id]

        # Por cada persona GT: oclusión + si fue detectada
        for pi in gt_person_idx:
            pbox = gt_boxes[pi]
            occ = max((iou(pbox, gt_boxes[j]) for j in range(len(gt_boxes)) if j != pi),
                      default=0.0)
            detected = any(iou(pbox, pred) >= IOU_MATCH for pred in pred_persons)
            persons.append((occ, detected))

    if not persons:
        raise SystemExit("No se encontraron personas en el test.")

    # Agrupar por terciles de oclusión
    persons.sort(key=lambda x: x[0])
    n = len(persons)
    cuts = [0, n // 3, 2 * n // 3, n]
    print(f"Total de personas (GT) en test: {n}\n")
    print(f"{'oclusión':<10}{'personas':>9}{'rango IoU':>16}{'recall':>9}")
    print("-" * 44)

    results = []
    for gi, group in enumerate(GROUPS):
        chunk = persons[cuts[gi]:cuts[gi + 1]]
        occs = [o for o, _ in chunk]
        rec = sum(1 for _, d in chunk if d) / len(chunk)
        rng = f"{min(occs):.3f}-{max(occs):.3f}"
        results.append((group, len(chunk), rng, rec))
        print(f"{group:<10}{len(chunk):>9}{rng:>16}{rec:>9.3f}")

    drop = results[0][3] - results[2][3]
    print(f"\nCaída de recall de oclusión baja -> alta: {drop:.3f} "
          f"({drop / results[0][3] * 100:.1f}% relativo)")

    # CSV
    csv = PROJECT_ROOT / "outputs" / "metrics" / "occlusion_person_recall.csv"
    csv.parent.mkdir(parents=True, exist_ok=True)
    with csv.open("w") as f:
        f.write("nivel_oclusion,personas,rango_iou,recall\n")
        for g, ni, rng, rec in results:
            f.write(f"{g},{ni},{rng},{rec:.4f}\n")
    print(f"CSV: {csv}")

    # Gráfico
    fig, ax = plt.subplots(figsize=(7, 4.5))
    ax.bar([r[0] for r in results], [r[3] for r in results],
           color=["#4caf50", "#ff9800", "#f44336"])
    ax.set_ylabel("Recall (personas detectadas)")
    ax.set_xlabel("Nivel de oclusión de la persona")
    ax.set_title("Recall en personas según su nivel de oclusión (test)")
    ax.set_ylim(0, 1)
    for i, r in enumerate(results):
        ax.text(i, r[3] + 0.02, f"{r[3]:.2f}", ha="center")
    plt.tight_layout()
    figp = PROJECT_ROOT / "report" / "figures" / "occlusion_recall.png"
    fig.savefig(figp, dpi=120)
    print(f"Gráfico: {figp}")


if __name__ == "__main__":
    main()


### 3.8 Robustez ante objetos pequeños

In [ ]:
%%writefile src/08_size_analysis.py
"""
Análisis de robustez ante objetos pequeños: recall en personas según su tamaño.

El análisis de oclusión (07) sugirió que el factor real de dificultad es el tamaño del objeto,
no la oclusión. Aquí se confirma de forma directa: las personas (ground truth) se agrupan por
tamaño relativo de su caja (pequeña/mediana/grande) y se mide el recall en cada grupo.

Tamaño relativo = área de la caja / área de la imagen.
Una persona se considera detectada si existe una predicción Person con IoU >= 0.5.

Genera:
    - outputs/metrics/size_person_recall.csv
    - report/figures/size_recall.png
"""

from pathlib import Path

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import yaml
from ultralytics import YOLO

PROJECT_ROOT = Path(__file__).resolve().parent.parent
DATASET_DIR = PROJECT_ROOT / "datasets" / "construction-site-safety"
DATA_YAML = DATASET_DIR / "data.yaml"
WEIGHTS = PROJECT_ROOT / "outputs" / "training" / "yolov8s_baseline" / "weights" / "best.pt"
TEST_IMAGES = DATASET_DIR / "test" / "images"
TEST_LABELS = DATASET_DIR / "test" / "labels"

CONF = 0.25
IOU_MATCH = 0.5
GROUPS = ["pequeña", "mediana", "grande"]


def iou(a, b):
    ax1, ay1, ax2, ay2 = a
    bx1, by1, bx2, by2 = b
    ix1, iy1 = max(ax1, bx1), max(ay1, by1)
    ix2, iy2 = min(ax2, bx2), min(ay2, by2)
    iw, ih = max(0.0, ix2 - ix1), max(0.0, iy2 - iy1)
    inter = iw * ih
    ua = (ax2 - ax1) * (ay2 - ay1) + (bx2 - bx1) * (by2 - by1) - inter
    return inter / ua if ua > 0 else 0.0


def main() -> None:
    if not WEIGHTS.exists():
        raise SystemExit(f"No existe el modelo: {WEIGHTS}")

    names = yaml.safe_load(DATA_YAML.read_text())["names"]
    person_id = names.index("Person")
    model = YOLO(str(WEIGHTS))

    persons = []  # (tamaño_relativo, detected)
    for img in sorted(TEST_IMAGES.glob("*.jpg")):
        label = TEST_LABELS / (img.stem + ".txt")
        if not label.exists():
            continue
        result = model(str(img), conf=CONF, verbose=False)[0]
        H, W = result.orig_shape

        pred_persons = [tuple(b.xyxy[0].tolist()) for b in result.boxes
                        if int(b.cls) == person_id]

        for line in label.read_text().splitlines():
            p = line.split()
            if len(p) < 5 or int(p[0]) != person_id:
                continue
            xc, yc, w, h = map(float, p[1:5])
            rel_size = w * h  # área relativa (w,h ya están normalizados 0-1)
            box = ((xc - w / 2) * W, (yc - h / 2) * H, (xc + w / 2) * W, (yc + h / 2) * H)
            detected = any(iou(box, pred) >= IOU_MATCH for pred in pred_persons)
            persons.append((rel_size, detected))

    persons.sort(key=lambda x: x[0])
    n = len(persons)
    cuts = [0, n // 3, 2 * n // 3, n]
    print(f"Total de personas (GT) en test: {n}\n")
    print(f"{'tamaño':<10}{'personas':>9}{'rango área':>18}{'recall':>9}")
    print("-" * 46)

    results = []
    for gi, group in enumerate(GROUPS):
        chunk = persons[cuts[gi]:cuts[gi + 1]]
        sizes = [s for s, _ in chunk]
        rec = sum(1 for _, d in chunk if d) / len(chunk)
        rng = f"{min(sizes):.4f}-{max(sizes):.4f}"
        results.append((group, len(chunk), rng, rec))
        print(f"{group:<10}{len(chunk):>9}{rng:>18}{rec:>9.3f}")

    gain = results[2][3] - results[0][3]
    print(f"\nDiferencia de recall pequeña -> grande: +{gain:.3f} "
          f"({gain / results[0][3] * 100:.1f}% relativo)")

    csv = PROJECT_ROOT / "outputs" / "metrics" / "size_person_recall.csv"
    csv.parent.mkdir(parents=True, exist_ok=True)
    with csv.open("w") as f:
        f.write("tamano,personas,rango_area_relativa,recall\n")
        for g, ni, rng, rec in results:
            f.write(f"{g},{ni},{rng},{rec:.4f}\n")
    print(f"CSV: {csv}")

    fig, ax = plt.subplots(figsize=(7, 4.5))
    ax.bar([r[0] for r in results], [r[3] for r in results],
           color=["#f44336", "#ff9800", "#4caf50"])
    ax.set_ylabel("Recall (personas detectadas)")
    ax.set_xlabel("Tamaño de la persona (área relativa)")
    ax.set_title("Recall en personas según su tamaño (test)")
    ax.set_ylim(0, 1)
    for i, r in enumerate(results):
        ax.text(i, r[3] + 0.02, f"{r[3]:.2f}", ha="center")
    plt.tight_layout()
    figp = PROJECT_ROOT / "report" / "figures" / "size_recall.png"
    fig.savefig(figp, dpi=120)
    print(f"Gráfico: {figp}")


if __name__ == "__main__":
    main()


### 3.9 Inferencia sobre video (opcional)

In [ ]:
%%writefile src/09_predict_video.py
"""
Inferencia del verificador de cumplimiento de EPP sobre un video (extensión de
06_predict_images.py a secuencias). Recorre el video fotograma a fotograma con OpenCV,
aplica el mismo modelo y módulo compliance_checker, y produce un video anotado, fotogramas
de muestra y un CSV con la tasa de cumplimiento por fotograma.

Uso (desde la raíz del repositorio):
    python src/09_predict_video.py                       # data/videos/demo.mp4 (por defecto)
    python src/09_predict_video.py data/videos/obra.mp4  # ruta explícita
    python src/09_predict_video.py --webcam              # cámara 0

Salida: cada fuente se guarda en su propia subcarpeta para no sobrescribir corridas previas:
    outputs/compliance/video/<nombre>/
        <nombre>_anotado.mp4        video completo anotado
        frames/frame_XXXXXX.jpg     fotogramas de muestra
        compliance_por_frame.csv    tasa de cumplimiento por fotograma
(La webcam usa la subcarpeta "webcam".)
"""

import sys
from pathlib import Path

import cv2
from ultralytics import YOLO

# Permite importar compliance_checker.py (está en la misma carpeta src/).
sys.path.insert(0, str(Path(__file__).resolve().parent))
from compliance_checker import PPEComplianceChecker, Detection, annotate_frame  # noqa: E402

# --------------------------------------------------------------------------------
# CONFIGURACIÓN (edítala aquí si lo necesitas)
# --------------------------------------------------------------------------------
PROJECT_ROOT = Path(__file__).resolve().parent.parent

CONFIG = {
    # Modelo entrenado (el mismo que usan los scripts 05 y 06).
    "weights": PROJECT_ROOT / "outputs" / "training" / "yolov8s_baseline" / "weights" / "best.pt",
    # Video de entrada por defecto (puedes sobreescribirlo por línea de comandos).
    "video": PROJECT_ROOT / "data" / "videos" / "demo.mp4",
    # Carpeta donde se guardan todas las salidas.
    "out_dir": PROJECT_ROOT / "outputs" / "compliance" / "video",
    # Umbral de confianza mínimo para quedarse con una detección.
    "conf": 0.25,
    # GPU (0) o CPU ("cpu").
    "device": 0,
    # Cada cuántos fotogramas se guarda uno como evidencia en frames/.
    # (No guardamos todos para no llenar el disco; 1 de cada 15 es razonable.)
    "save_every": 15,
}


def detections_from_result(result, names) -> list:
    """Convierte el resultado de YOLO de UN fotograma en una lista de Detection.

    Idéntico a 06_predict_images.py: por cada caja toma su clase, confianza y las
    coordenadas xyxy en píxeles.
    """
    dets = []
    if result.boxes is None:
        return dets
    for box in result.boxes:
        cls_id = int(box.cls)
        dets.append(Detection(
            cls_name=names[cls_id],
            conf=float(box.conf),
            xyxy=tuple(float(v) for v in box.xyxy[0].tolist()),
        ))
    return dets


def main() -> None:
    if not CONFIG["weights"].exists():
        raise SystemExit(
            f"No existe el modelo: {CONFIG['weights']}\n"
            "Entrena primero: python src/04_train.py"
        )

    # 1) Resolver la fuente del video (archivo o webcam) desde la línea de comandos.
    use_webcam = "--webcam" in sys.argv
    extra_args = [a for a in sys.argv[1:] if not a.startswith("--")]
    video_path = Path(extra_args[0]) if extra_args else CONFIG["video"]
    source = 0 if use_webcam else str(video_path)

    if not use_webcam and not video_path.exists():
        print(f"[ERROR] No encuentro el video: {video_path}")
        print("        Copia tu archivo en data/videos/ o pasa la ruta como argumento:")
        print("        ../.venv/bin/python src/09_predict_video.py data/videos/tu_video.mp4")
        sys.exit(1)

    # 2) Preparar carpetas de salida.
    #    Cada fuente tiene su PROPIA subcarpeta, para no sobrescribir resultados de otros
    #    videos: outputs/compliance/video/<nombre_del_video>/. Para la webcam usamos un
    #    nombre fijo ("webcam"). Así puedes correr video1, video2, video3... y conservar
    #    la evidencia de cada uno por separado.
    run_name = "webcam" if use_webcam else video_path.stem
    out_dir = CONFIG["out_dir"] / run_name
    frames_dir = out_dir / "frames"
    frames_dir.mkdir(parents=True, exist_ok=True)
    print(f"[INFO] Salida de esta corrida: {out_dir}")

    # 3) Cargar el modelo y el verificador (misma configuración que 06).
    print(f"[INFO] Cargando modelo: {CONFIG['weights']}")
    model = YOLO(str(CONFIG["weights"]))
    names = model.names
    checker = PPEComplianceChecker(min_conf=CONFIG["conf"])

    # 4) Abrir el video con OpenCV.
    cap = cv2.VideoCapture(source)
    if not cap.isOpened():
        print(f"[ERROR] No pude abrir la fuente de video: {source}")
        sys.exit(1)

    fps = cap.get(cv2.CAP_PROP_FPS) or 25.0
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT)) if not use_webcam else -1
    print(f"[INFO] Video: {width}x{height} @ {fps:.1f} fps, {total} fotogramas")

    # 5) Preparar el escritor del video anotado (mp4). Lleva el nombre de la fuente para
    #    identificarlo fácilmente al descargarlo (ej. video1_anotado.mp4).
    out_video_path = out_dir / f"{run_name}_anotado.mp4"
    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    writer = cv2.VideoWriter(str(out_video_path), fourcc, fps, (width, height))

    # 6) Preparar el CSV de evidencia (tasa de cumplimiento por fotograma).
    csv_path = out_dir / "compliance_por_frame.csv"
    csv_file = open(csv_path, "w", encoding="utf-8")
    csv_file.write("frame,personas,cumplen,tasa_cumplimiento\n")

    # 7) Bucle principal: leer -> detectar -> verificar -> anotar -> escribir.
    frame_idx = 0
    total_personas = 0
    total_cumplen = 0
    try:
        while True:
            ok, frame = cap.read()
            if not ok:
                break  # se acabó el video

            result = model.predict(frame, conf=CONFIG["conf"],
                                   device=CONFIG["device"], verbose=False)[0]
            dets = detections_from_result(result, names)

            # check_frame devuelve (lista de PersonStatus, tasa[0..1] o None si no hay personas)
            statuses, rate = checker.check_frame(dets)
            n_personas = len(statuses)
            n_ok = sum(1 for st in statuses if st.compliant)
            total_personas += n_personas
            total_cumplen += n_ok
            tasa_txt = f"{rate:.3f}" if rate is not None else ""
            csv_file.write(f"{frame_idx},{n_personas},{n_ok},{tasa_txt}\n")

            # annotate_frame dibuja cajas verdes/rojas + banner y devuelve el fotograma.
            annotated = annotate_frame(frame, statuses, rate)
            writer.write(annotated)

            # Guardar un fotograma de muestra como evidencia ligera.
            if frame_idx % CONFIG["save_every"] == 0:
                cv2.imwrite(str(frames_dir / f"frame_{frame_idx:06d}.jpg"), annotated)

            frame_idx += 1
            if frame_idx % 50 == 0:
                print(f"[INFO] Procesados {frame_idx} fotogramas...")
    finally:
        cap.release()
        writer.release()
        csv_file.close()

    # 8) Resumen final.
    print("\n========== RESUMEN ==========")
    print(f"Fotogramas procesados : {frame_idx}")
    if total_personas:
        tasa_global = 100 * total_cumplen / total_personas
        print(f"Personas (acumulado)  : {total_personas}")
        print(f"Cumplen (acumulado)   : {total_cumplen}")
        print(f"Tasa de cumplimiento  : {tasa_global:.1f}%")
    print(f"Video anotado         : {out_video_path}")
    print(f"Fotogramas evidencia  : {frames_dir}/")
    print(f"CSV por fotograma     : {csv_path}")


if __name__ == "__main__":
    main()


## 4. Ejecución del pipeline

Cada paso usa los archivos de código de la sección anterior. El entrenamiento (paso 4.4) tarda
unos 30-40 minutos en GPU.


In [ ]:
!python src/01_download_dataset.py

In [ ]:
!python src/02_analyze_dataset.py

In [ ]:
!python src/03_visualize_samples.py

In [ ]:
!python src/04_train.py

In [ ]:
!python src/05_validate.py

In [ ]:
!python src/06_predict_images.py

In [ ]:
!python src/07_occlusion_analysis.py

In [ ]:
!python src/08_size_analysis.py

## 5. Resultados

In [ ]:
import os, glob
from IPython.display import Image, display
for p in ["outputs/training/yolov8s_baseline/results.png",
          "outputs/training/yolov8s_baseline/confusion_matrix.png",
          "report/figures/demo_cumplimiento.png",
          "report/figures/occlusion_recall.png",
          "report/figures/size_recall.png"]:
    if os.path.exists(p): print(p); display(Image(p))
for p in sorted(glob.glob("outputs/compliance/images/*.jpg"))[:3]:
    display(Image(p))
print("\nMétricas por clase:")
print(open("outputs/metrics/test_metrics_per_class.csv").read())


## 6. Inferencia sobre un video (opcional)

Sube un archivo `.mp4` y ejecuta el verificador sobre él:


In [ ]:
import os
from google.colab import files
os.makedirs("data/videos", exist_ok=True)
for fn in files.upload():
    os.replace(fn, f"data/videos/{fn}")
    !python src/09_predict_video.py "data/videos/{fn}"
